# REMNET Stage 3 — D3 External Inference Sustainability (Classification Checkpoints)

**Source references:** `remnet-co2-prob-s2.ipynb` (Stage 2 CodeCarbon monitoring) and `remnet-class-d3.ipynb` (Stage 3 D3-only frozen-checkpoint loading, RealWaste D3 manifest verification, dataset resolution, and equal-weight ensemble).

Five seeds: **7, 49, 112, 345, 476**. All runs use the **three seed-matched REMNET classification checkpoints** (MobileNetV2, ResNet50, EfficientNet-Lite0) with strict state-dict loading. No training, adaptation, or checkpoint modification. D3 contains exactly **3,497** screened RealWaste images, preprocessing **RGB → 484 × 484 → tensor → normalize (0.5,0.5,0.5)**.

CodeCarbon is used to monitor **D3 external inference only**; original training energy is not remeasured and D1 original-test evaluation is not performed. Each complete repeated pass evaluates the same 3,497 D3 images; model load, data checks and warm-up are excluded. Original CodeCarbon CO₂ is reported separately from standardized CO₂ using the original Stage 2 factor 0.475 kg/kWh.

**CO₂e reporting:** Inference energy is multiplied by the declared fixed factor of **0.475 kg CO₂e/kWh** for a standardized comparison with Stage 2, while the original CodeCarbon `emissions` output is retained as an independent column and original log. Both quantities appear in printed summaries and the ZIP CSVs.


## 1. Locked experiment inputs and output configuration

In [5]:
from __future__ import annotations
import os, sys, json, hashlib, zipfile, subprocess, importlib.util, traceback
from pathlib import Path, PurePosixPath
from collections import Counter
from datetime import datetime, timezone

# Kaggle input location confirmed by the user; adjust ONLY if Kaggle moves the dataset.
INPUT_ROOT = Path('/kaggle/input/datasets/akash264208/manifist-part2/Section3')
OUTPUT = Path('/kaggle/working/REMNET_D3_Inference_Sustainability')
OUTPUT.mkdir(parents=True, exist_ok=True)
SEEDS = (7, 49, 112, 345, 476)
MODEL_NAME = 'REMNET'
CARBON_FACTOR = 0.475  # kg CO2e/kWh, fixed assumption from Stage 2
MIN_COMPLETE_PASSES = 2
MIN_MONITOR_SECONDS = 35.0
CODECARBON_SAMPLING_SECONDS = 15
ZIP_PATH = Path('/kaggle/working/REMNET_D3_Inference_Sustainability_All_Outputs.zip')
CONSTITUENTS = ('MobileNetV2','ResNet50','EfficientNet-Lite0')
CLASSES = ('cardboard','glass','metal','paper','plastic','trash')
CLASS_TO_ID = {name:i for i,name in enumerate(CLASSES)}
EXPECTED_EXTERNAL_COUNT = 3497
EXPECTED_EXTERNAL_MANIFEST_FINGERPRINT = '0350db99b6be93bfaa9a1dfcdaf99d9c127404658a6222c722bc83fb131cf0e0'
# Set this to the exact Kaggle folder containing the D3 Section 2 CSV files.
# Example: '/kaggle/input/datasets/YOUR_ACCOUNT/YOUR_D3_SECTION2_OUTPUTS'
# If None, the notebook locates unique matching files under /kaggle/input.
D3_MANIFEST_ROOT = '/kaggle/input/datasets/akash264208/manifist-part2/Section3/manifist'
BATCH_SIZE = 32

# Optional genuine historical Stage 2 REMNET ensemble prediction files.
# Each file, when present, must have source_path and predicted_class_id columns,
# and preferably all six prob_CLASS columns. They must come from Stage 2.
# Set None to discover files automatically in the attached private Kaggle input.
HISTORICAL_REFERENCE_DIR = None
REQUIRE_HISTORICAL_REFERENCE = False  # Set True to STOP external evaluation if references are absent.

assert INPUT_ROOT.is_dir(), f'Input dataset was not attached: {INPUT_ROOT}'
print('Kaggle input:', INPUT_ROOT)
print('Output directory:', OUTPUT)
print('Seeds:', SEEDS)
print('Frozen classification checkpoints, Stage 3 D3 sustainability; no training or fine-tuning.')

Kaggle input: /kaggle/input/datasets/akash264208/manifist-part2/Section3
Output directory: /kaggle/working/REMNET_D3_Inference_Sustainability
Seeds: (7, 49, 112, 345, 476)
Frozen classification checkpoints, Stage 3 D3 sustainability; no training or fine-tuning.


## 2. Dependencies and strict classification-checkpoint inventory

In [6]:
# Load dependencies without downloading pretrained weights.
for name in ('kagglehub', 'timm'):
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', name])
import kagglehub, timm, re, numpy as np, pandas as pd, torch
from torch import nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CPU inference selected')
print('torch:', torch.__version__, 'timm:', timm.__version__)

def only_directory_with_name(root, name):
    matches = sorted(p for p in root.rglob('*') if p.is_dir() and p.name.casefold() == name.casefold())
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one directory {name!r} under {root}; found {matches}')
    return matches[0]

def unique_by_filename(root, filename):
    matches = sorted(p for p in root.rglob(filename) if p.is_file())
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one {filename!r} in {root}: {matches}')
    return matches[0]

CKPT_ROOT = only_directory_with_name(INPUT_ROOT, 'Classification_checkpoints')
REMNET_CKPT_DIR = only_directory_with_name(CKPT_ROOT, 'Remnet')
STANDALONE_CKPT_DIR = only_directory_with_name(CKPT_ROOT, 'Resnet_50')
RESULTS_ROOT = only_directory_with_name(INPUT_ROOT, 'per_seed_performance_summary')
# D3 Section 2 manifest and class mapping are read directly from the user's Kaggle input.
# There is no embedded or reconstructed manifest.
def d3_csv_path(filename):
    root = Path(D3_MANIFEST_ROOT) if D3_MANIFEST_ROOT else Path('/kaggle/input')
    if not root.is_dir():
        raise FileNotFoundError(f'D3 manifest folder not attached: {root}')
    return unique_by_filename(root, filename)

EXTERNAL_MANIFEST = d3_csv_path('D3_Section2_Final_Automated_Screened_Six_Class_Manifest.csv')
CLASS_MAPPING = d3_csv_path('D3_Section2_Class_Mapping.csv')

CHECKPOINTS = {}
for seed in SEEDS:
    CHECKPOINTS[seed] = {}
    for name in CONSTITUENTS:
        # Exact original Stage 2 filenames, with optional Kaggle duplicate suffix (1).
        pattern = re.compile(rf'{re.escape(name)}_seed{seed}(?:\(\d+\))?\.pt', re.IGNORECASE)
        found = sorted(p for p in REMNET_CKPT_DIR.rglob('*.pt') if pattern.fullmatch(p.name))
        if len(found) != 1:
            raise RuntimeError(f'Expected one REMNET checkpoint for {name} seed {seed}, found: {found}')
        CHECKPOINTS[seed][name] = found[0]

all_pts = sorted(CKPT_ROOT.rglob('*.pt'))
remnet_pts = sorted(REMNET_CKPT_DIR.rglob('*.pt'))
assert len(remnet_pts) == 15, f'Expected 15 REMNET constituent checkpoints, found {len(remnet_pts)}'
assert len({p for group in CHECKPOINTS.values() for p in group.values()}) == 15
assert len(all_pts) == 30, f'Expected 30 total Stage 2 checkpoints, found {len(all_pts)}'
assert not any(p.is_relative_to(STANDALONE_CKPT_DIR) for group in CHECKPOINTS.values() for p in group.values())

# Stage 2 REMNET-only historical references; do not use standalone ResNet50 reports.
HISTORICAL_PER_SEED = next(iter(sorted(RESULTS_ROOT.rglob('REMNET_Performance_Per_Seed.csv'))), None)
HISTORICAL_SUMMARY = next(iter(sorted(RESULTS_ROOT.rglob('REMNET_5_Seed_Performance_Summary.csv'))), None)
print('Total .pt checkpoints:', len(all_pts), '| REMNET constituent checkpoints:', len(remnet_pts))
for seed, group in CHECKPOINTS.items():
    for name, path in group.items(): print('REMNET', seed, name, ':', path)
for path in [EXTERNAL_MANIFEST, CLASS_MAPPING]: print('CSV:', path)
print('Historical REMNET per-seed:', HISTORICAL_PER_SEED or 'not present')
print('Historical REMNET summary:', HISTORICAL_SUMMARY or 'not present')

# Monitoring follows Stage 2 CodeCarbon 3.3.2. Install only if necessary.
from importlib.metadata import version as installed_version
try:
    _cc_version = installed_version('codecarbon')
except Exception:
    _cc_version = None
if _cc_version != '3.3.2':
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'codecarbon==3.3.2'])
from codecarbon import EmissionsTracker
import time, shutil, platform
print('CodeCarbon:',installed_version('codecarbon'))


GPU: Tesla T4
torch: 2.11.0+cu130 timm: 1.0.30
Total .pt checkpoints: 30 | REMNET constituent checkpoints: 15
REMNET 7 MobileNetV2 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/Remnet/MobileNetV2_seed7.pt
REMNET 7 ResNet50 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/Remnet/ResNet50_seed7.pt
REMNET 7 EfficientNet-Lite0 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/Remnet/EfficientNet-Lite0_seed7.pt
REMNET 49 MobileNetV2 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/Remnet/MobileNetV2_seed49.pt
REMNET 49 ResNet50 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/Remnet/ResNet50_seed49.pt
REMNET 49 EfficientNet-Lite0 : /kaggle/input/datasets/akash264208/manifist-part2/Section3/Classification_checkpoints/Remnet/EfficientNet-Lite0_seed49.pt
REMNET 112 MobileNetV2 : /kaggle/input/dataset

## 3. Validate the locked D3 manifest and six-class RealWaste mapping

In [7]:
def file_sha256(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path,'rb') as stream:
        for b in iter(lambda:stream.read(chunk_size),b''):
            h.update(b)
    return h.hexdigest()

def canonical_external_fingerprint(df):
    h = hashlib.sha256()
    for row in df.sort_values('external_relative_path').itertuples(index=False):
        h.update((row.external_relative_path+'|'+str(int(row.output_index))+'|'+
                  row.sha256+'\n').encode('utf-8'))
    return h.hexdigest()

external_df = pd.read_csv(EXTERNAL_MANIFEST, dtype=str, keep_default_na=False)
mapping_df = pd.read_csv(CLASS_MAPPING, dtype=str, keep_default_na=False)

assert set(['external_relative_path','class','output_index','sha256','final_status']) <= set(external_df.columns)
assert set(['d3_source_label','remnet_trained_class','remnet_output_index','include_for_primary_d3']) <= set(mapping_df.columns)
assert len(external_df)==EXPECTED_EXTERNAL_COUNT, 'Unexpected D3 manifest count (should be 3497).'
assert external_df['sha256'].nunique()==EXPECTED_EXTERNAL_COUNT, 'Duplicate hashes in external manifest.'
assert external_df['external_relative_path'].nunique()==EXPECTED_EXTERNAL_COUNT
assert set(external_df['class']) == set(CLASSES)
assert external_df['final_status'].eq('ELIGIBLE_AFTER_REFERENCE_VERIFICATION').all(), 'Noneligible images in frozen manifest.'
included_mapping = mapping_df.loc[mapping_df['include_for_primary_d3'].str.lower().eq('true')].copy()
assert len(included_mapping) == 6
assert set(included_mapping['remnet_trained_class']) == set(CLASSES)
for item in included_mapping.itertuples(index=False):
    assert int(item.remnet_output_index) == CLASS_TO_ID[item.remnet_trained_class]
    if 'class_review' in mapping_df.columns:
        assert item.class_review == 'CONFIRMED_BY_USER'
assert set(external_df['source_category']) == set(included_mapping['d3_source_label'])
assert all(int(i)==CLASS_TO_ID[c] for c,i in zip(external_df['class'],external_df['output_index']))

EXTERNAL_FINGERPRINT=canonical_external_fingerprint(external_df)
assert EXTERNAL_FINGERPRINT == EXPECTED_EXTERNAL_MANIFEST_FINGERPRINT, (EXTERNAL_FINGERPRINT,'External cohort fingerprint mismatch')
print('D3 frozen manifest verified:',len(external_df),'Fingerprint:',EXTERNAL_FINGERPRINT)
print('Six class mapping:',CLASS_TO_ID)
print('External per-class counts:',external_df['class'].value_counts().sort_index().to_dict())

D3 frozen manifest verified: 3497 Fingerprint: 0350db99b6be93bfaa9a1dfcdaf99d9c127404658a6222c722bc83fb131cf0e0
Six class mapping: {'cardboard': 0, 'glass': 1, 'metal': 2, 'paper': 3, 'plastic': 4, 'trash': 5}
External per-class counts: {'cardboard': 454, 'glass': 419, 'metal': 757, 'paper': 500, 'plastic': 889, 'trash': 478}


## 4. Load the attached RealWaste D3 dataset

In [8]:
# Stage 3 D3 dataset path — fixed mounted Kaggle input.
EXTERNAL_HANDLE = 'akash264208/real-waste'
external_download = Path('/kaggle/input/datasets/akash264208/real-waste')
external_download_method = 'attached Kaggle dataset'
assert external_download.is_dir(), f'D3 RealWaste dataset missing: {external_download}'
print('D3 RealWaste:', external_download)

D3 RealWaste: /kaggle/input/datasets/akash264208/real-waste


## 5. Resolve D3 images and verify every locked image hash

In [9]:
IMG_EXT={'.jpg','.jpeg','.png','.webp','.bmp'}

def safe_parts(relative):
    p=PurePosixPath(str(relative).replace('\\','/'))
    if p.is_absolute() or not p.parts or any(s in ('','.','..') for s in p.parts):
        raise ValueError(f'Unsafe image relative path: {relative!r}')
    return p.parts

def external_class_roots(root):
    expected=set(included_mapping['d3_source_label'])
    candidates=[root] + sorted((p for p in root.rglob('*') if p.is_dir()), key=str)
    return [p for p in candidates if all((p/c).is_dir() for c in expected)]

def resolve_external(external,download):
    candidates=external_class_roots(download)
    print('D3 RealWaste class-root candidates:',candidates)
    if not candidates: raise FileNotFoundError('Missing RealWaste source-category directories.')
    expected_rows=[(safe_parts(r.external_relative_path),r.sha256,r.source_category)
                   for _,r in external.iterrows()]
    for root in candidates:
        try:
            dirs=[p for p in root.iterdir() if p.is_dir()]
            image_counts={p.name:sum(x.is_file() and x.suffix.lower() in IMG_EXT
                         for x in p.rglob('*')) for p in dirs}
            if len(image_counts)!=9 or sum(image_counts.values())!=4752:
                raise RuntimeError(f'Expected nine RealWaste folders / 4752 images: {image_counts}')
            if sum(image_counts[c] for c in included_mapping['d3_source_label'])!=3587:
                raise RuntimeError('D3 six-category source total differs from Section 2.')
            resolved=[]
            for parts,digest,source_cls in expected_rows:
                image=root.joinpath(*parts)
                if not image.is_file() or parts[0]!=source_cls:
                    raise FileNotFoundError(str(image))
                if file_sha256(image)!=digest:
                    raise ValueError(f'D3 image hash mismatch: {image}')
                resolved.append(str(image))
            print('D3 RealWaste images SHA-256 verified:',len(resolved),'root:',root)
            return resolved,root,image_counts
        except (FileNotFoundError,ValueError,RuntimeError) as exc:
            print('D3 candidate rejected:',root,'reason:',str(exc)[:180])
    raise RuntimeError('No RealWaste root matches the 3497 locked image hashes.')

external_paths, external_image_root, d3_counts=resolve_external(external_df,external_download)
external_df=external_df.copy(); external_df['resolved_path']=external_paths
external_eval_df=external_df.sort_values('external_relative_path').reset_index(drop=True)
assert len(external_eval_df)==3497
print('External evaluated images:',len(external_eval_df))

D3 RealWaste class-root candidates: [PosixPath('/kaggle/input/datasets/akash264208/real-waste/realwaste-main/RealWaste')]
D3 RealWaste images SHA-256 verified: 3497 root: /kaggle/input/datasets/akash264208/real-waste/realwaste-main/RealWaste
External evaluated images: 3497


## 6. Frozen REMNET ensemble model and preprocessing

In [10]:
TEST_TRANSFORM = transforms.Compose([
    transforms.Resize((484,484)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3,(0.5,)*3),
])

class LockedImageDataset(Dataset):
    def __init__(self, df, kind):
        self.df = df.reset_index(drop=True)
        self.path_col = 'external_relative_path'
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        with Image.open(row['resolved_path']) as im:
            image = TEST_TRANSFORM(im.convert('RGB'))
        return image, int(CLASS_TO_ID[row['class']]), i

def locked_loader(df,kind):
    return DataLoader(LockedImageDataset(df,kind),batch_size=BATCH_SIZE,
                      shuffle=False,num_workers=0,pin_memory=(DEVICE.type=='cuda'))

def construct_constituent(name):
    n = len(CLASSES)
    if name == 'MobileNetV2':
        m = models.mobilenet_v2(weights=None)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, n)
    elif name == 'ResNet50':
        m = models.resnet50(weights=None)
        m.fc = nn.Linear(m.fc.in_features, n)
    elif name == 'EfficientNet-Lite0':
        m = timm.create_model('efficientnet_lite0', pretrained=False, num_classes=n)
    else:
        raise ValueError(name)
    return m

def restore_checkpoint(seed):
    ensemble, metadata, digests = {}, {}, {}
    for name in CONSTITUENTS:
        path = CHECKPOINTS[seed][name]
        obj = torch.load(path,map_location='cpu',weights_only=True)
        assert isinstance(obj,dict) and 'state_dict' in obj, f'Unsupported Stage 2 checkpoint: {path}'
        assert str(obj.get('model')) == name, f'Constituent mismatch in {path.name}: {obj.get("model")}'
        assert int(obj.get('seed')) == seed, f'Seed mismatch in {path.name}'
        m = construct_constituent(name)
        m.load_state_dict(obj['state_dict'],strict=True)
        m.to(DEVICE).eval()
        for param in m.parameters(): param.requires_grad_(False)
        ensemble[name],metadata[name],digests[name] = m,obj,file_sha256(path)
    return ensemble,metadata,digests

def infer(ensemble,df,kind):
    labels, probabilities, positions = [],[],[]
    with torch.inference_mode():
        for images,y,index in locked_loader(df,kind):
            images=images.to(DEVICE,non_blocking=True)
            # Exact Stage 2 REMNET rule: arithmetic mean of three softmax vectors.
            stacked = torch.stack([torch.softmax(ensemble[name](images),dim=1)
                                   for name in CONSTITUENTS],dim=0)
            prob=stacked.mean(dim=0)
            probabilities.append(prob.cpu().numpy().astype(np.float64))
            labels.append(y.numpy()); positions.append(index.numpy())
    labels=np.concatenate(labels).astype(np.int64)
    probabilities=np.concatenate(probabilities)
    positions=np.concatenate(positions)
    assert np.array_equal(positions,np.arange(len(df))), 'Image order changed during inference.'
    assert np.array_equal(labels,df['class'].map(CLASS_TO_ID).to_numpy()), 'Label order mismatch.'
    assert probabilities.shape==(len(df),6) and np.isfinite(probabilities).all()
    assert np.allclose(probabilities.sum(axis=1),1,atol=2e-6)
    return labels,probabilities

_smoke, _smoke_meta, _smoke_hash = restore_checkpoint(SEEDS[0])
with torch.inference_mode():
    batch=torch.zeros(1,3,484,484,device=DEVICE)
    assert all(tuple(_smoke[name](batch).shape)==(1,6) for name in CONSTITUENTS)
print('REMNET: all three exact Stage 2 constituent architectures strictly loaded; six-logit smoke test PASS')
del _smoke,_smoke_meta,_smoke_hash,batch
if DEVICE.type=='cuda':torch.cuda.empty_cache()


REMNET: all three exact Stage 2 constituent architectures strictly loaded; six-logit smoke test PASS


## 7. CodeCarbon and timing protocol

Warm up **all three** constituents before monitoring. During monitoring, every image is classified through the complete equal-weight soft-voting ensemble. Synchronize the GPU around timing. Use at least two complete passes and at least 35 seconds of monitoring to provide more than two 15-second sampling intervals. The exact number of passes is logged. These are **inference-workload** measurements, including data loading and preprocessing within the timed loop; model construction, checkpoint loading and warm-up are excluded. No D1 inference is included in CodeCarbon monitoring.

In [11]:
LOG_ROOT = OUTPUT/'CodeCarbon_Logs'
LOG_ROOT.mkdir(parents=True,exist_ok=True)

def make_tracker(log_dir, seed):
    log_dir.mkdir(parents=True,exist_ok=True)
    prior = log_dir/'emissions.csv'
    if prior.exists():
        prior.rename(log_dir/f'emissions_previous_{time.time_ns()}.csv')
    options = dict(
        project_name=f'REMNET_Stage3_seed{seed}_External3497',
        output_dir=str(log_dir), output_file='emissions.csv',
        save_to_file=True, tracking_mode='machine',
        measure_power_secs=CODECARBON_SAMPLING_SECONDS,
        pue=1.0, log_level='error'
    )
    if DEVICE.type=='cuda': options['gpu_ids']=[0]
    return EmissionsTracker(**options)

def gpu_sync():
    if DEVICE.type=='cuda': torch.cuda.synchronize()

def measured_row(log_dir):
    filepath=log_dir/'emissions.csv'
    if not filepath.is_file():
        raise FileNotFoundError(f'CodeCarbon emissions file missing: {filepath}')
    df=pd.read_csv(filepath)
    if df.empty: raise RuntimeError(f'Empty CodeCarbon file: {filepath}')
    row=df.iloc[-1].to_dict()
    energy=float(row['energy_consumed'])
    if not np.isfinite(energy) or energy<0: raise RuntimeError(f'Invalid CodeCarbon energy {energy}')
    emissions=float(row['emissions']) if 'emissions' in row and pd.notna(row['emissions']) else np.nan
    return row,energy,emissions

def monitored_remnet_inference(seed, ensemble):
    loader=locked_loader(external_eval_df,'external')
    for model in ensemble.values(): model.eval()
    # The same whole-ensemble work is performed before the clock starts.
    images, _, _ = next(iter(loader))
    images=images.to(DEVICE,non_blocking=True)
    with torch.inference_mode():
        for _ in range(2):
            p=torch.stack([torch.softmax(ensemble[name](images),dim=1)
                           for name in CONSTITUENTS],dim=0).mean(dim=0)
            if p.shape[1]!=6:raise RuntimeError('Unexpected output size in warm-up')
    gpu_sync()
    del images,p
    log_dir=LOG_ROOT/f'REMNET_seed{seed}'/'inference'
    tracker=make_tracker(log_dir,seed)
    passes=0
    processed=0
    labels_first=[]; pred_first=[]
    started=False
    try:
        tracker.start()
        started=True
        gpu_sync()
        t0=time.perf_counter()
        with torch.inference_mode():
            while passes<MIN_COMPLETE_PASSES or (time.perf_counter()-t0)<MIN_MONITOR_SECONDS:
                current_positions=[]
                for images,y,positions in loader:
                    images=images.to(DEVICE,non_blocking=True)
                    probs=torch.stack([torch.softmax(ensemble[name](images),dim=1)
                                       for name in CONSTITUENTS],dim=0).mean(dim=0)
                    if probs.shape!=(len(y),6):raise RuntimeError('Six-class output mismatch')
                    if passes==0:
                        labels_first.extend(y.tolist())
                        pred_first.extend(probs.argmax(dim=1).cpu().tolist())
                        current_positions.extend(positions.tolist())
                    processed += len(y)
                if passes==0 and current_positions!=list(range(len(external_eval_df))):
                    raise RuntimeError('External image order changed during monitored inference')
                passes += 1
        gpu_sync()
        seconds=time.perf_counter()-t0
    finally:
        if started:
            gpu_sync()
            tracker.stop()
    if processed!=passes*len(external_eval_df):
        raise RuntimeError(f'Incomplete inference pass: {processed} images / {passes} passes')
    row,energy,cc_emissions=measured_row(log_dir)
    if seconds<=0 or processed<=0:raise RuntimeError('Invalid measured workload')
    ytrue=np.asarray(labels_first,dtype=int)
    ypred=np.asarray(pred_first,dtype=int)
    expected=external_eval_df['class'].map(CLASS_TO_ID).to_numpy()
    assert np.array_equal(ytrue,expected)
    accuracy_pct=100*float((ytrue==ypred).mean())
    result={
       'model':'REMNET','seed':int(seed),'unique_external_images':len(external_eval_df),
       'number_complete_repetitions':passes,'total_processed_images':processed,
       'batch_size':BATCH_SIZE,'input_resolution':'484x484',
       'monitoring_seconds':seconds,'total_inference_energy_kWh':energy,
       'codecarbon_emissions_kgCO2e':cc_emissions,
       'standardized_CO2e_kg_fixed_factor':energy*CARBON_FACTOR,
       'standardized_CO2_factor_kg_per_kWh':CARBON_FACTOR,
       'energy_kWh_per_processed_image':energy/processed,
       'time_seconds_per_processed_image':seconds/processed,
       'throughput_images_per_second':processed/seconds,
       'first_pass_accuracy_pct_diagnostic':accuracy_pct,
       'codecarbon_duration_seconds':row.get('duration',np.nan),
       'codecarbon_energy_column':'energy_consumed',
       'codecarbon_version':installed_version('codecarbon'),
       'gpu':torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else 'CPU',
       'raw_emissions_csv':str(log_dir/'emissions.csv')
    }
    return result


## 8. Five-seed external inference monitoring

Each seed loads **three REMNET classification checkpoints** and records their individual file SHA-256 values. All five seeds are evaluated against the identical locked D3 external manifest. Checkpoint hashes can be matched against the Stage 3 D3 classification `REMNET_Checkpoint_Registry.csv`.

In [12]:
sustainability_rows=[]
checkpoint_rows=[]
for seed in SEEDS:
    print(f'\n=== REMNET STAGE 3 D3 INFERENCE SUSTAINABILITY — SEED {seed} ===',flush=True)
    ensemble,meta,digests=restore_checkpoint(seed)
    for name in CONSTITUENTS:
        checkpoint_rows.append({
            'seed':seed,'model':'REMNET','constituent':name,
            'checkpoint_path':str(CHECKPOINTS[seed][name]),
            'checkpoint_file_sha256':digests[name],
            'checkpoint_seed':meta[name].get('seed'),
            'split_sha256':meta[name].get('split_sha256'),
            'best_epoch':meta[name].get('best_epoch'),
            'frozen':True})
    row=monitored_remnet_inference(seed,ensemble)
    sustainability_rows.append(row)
    pd.DataFrame(sustainability_rows).to_csv(OUTPUT/'REMNET_Stage3_D3_Inference_Per_Seed.csv',index=False)
    pd.DataFrame(checkpoint_rows).to_csv(OUTPUT/'REMNET_Stage3_D3_Inference_Checkpoint_Registry.csv',index=False)
    print('D3 unique:',row['unique_external_images'],'| passes:',row['number_complete_repetitions'],
          '| total processed:',row['total_processed_images'])
    print('Energy (kWh):',f"{row['total_inference_energy_kWh']:.9f}",
          '| CodeCarbon CO2e (kg):',f"{row['codecarbon_emissions_kgCO2e']:.9f}",
          '| Standardized CO2e at 0.475 (kg):',f"{row['standardized_CO2e_kg_fixed_factor']:.9f}",
          '| Runtime (s):',f"{row['monitoring_seconds']:.3f}",
          '| Energy/image (kWh):',f"{row['energy_kWh_per_processed_image']:.12f}",
          '| Throughput (img/s):',f"{row['throughput_images_per_second']:.3f}")
    del ensemble
    if DEVICE.type=='cuda':torch.cuda.empty_cache()
print('Completed',len(sustainability_rows),'seeds')



=== REMNET STAGE 3 D3 INFERENCE SUSTAINABILITY — SEED 7 ===


[codecarbon WARNING @ 11:33:27] Multiple instances of codecarbon are allowed to run at the same time.


D3 unique: 3497 | passes: 2 | total processed: 6994
Energy (kWh): 0.000581083 | CodeCarbon CO2e (kg): 0.000147698 | Standardized CO2e at 0.475 (kg): 0.000276014 | Runtime (s): 40.975 | Energy/image (kWh): 0.000000083083 | Throughput (img/s): 170.688

=== REMNET STAGE 3 D3 INFERENCE SUSTAINABILITY — SEED 49 ===
D3 unique: 3497 | passes: 2 | total processed: 6994
Energy (kWh): 0.000591776 | CodeCarbon CO2e (kg): 0.000150416 | Standardized CO2e at 0.475 (kg): 0.000281094 | Runtime (s): 41.182 | Energy/image (kWh): 0.000000084612 | Throughput (img/s): 169.830

=== REMNET STAGE 3 D3 INFERENCE SUSTAINABILITY — SEED 112 ===
D3 unique: 3497 | passes: 2 | total processed: 6994
Energy (kWh): 0.000588847 | CodeCarbon CO2e (kg): 0.000149671 | Standardized CO2e at 0.475 (kg): 0.000279702 | Runtime (s): 40.527 | Energy/image (kWh): 0.000000084193 | Throughput (img/s): 172.577

=== REMNET STAGE 3 D3 INFERENCE SUSTAINABILITY — SEED 345 ===
D3 unique: 3497 | passes: 2 | total processed: 6994
Energy (kW

## 9. Five-seed mean ± sample SD and measurement provenance

Sample standard deviations use `ddof=1`. Per-image values use **total processed images**, which include every monitored repetition. Total inference energy and original CodeCarbon CO₂ are separate from any fixed-factor estimates.

In [13]:
results_df=pd.DataFrame(sustainability_rows)
assert len(results_df)==5 and set(results_df.seed)==set(SEEDS)
MEASURES=[
    'total_inference_energy_kWh','codecarbon_emissions_kgCO2e',
    'standardized_CO2e_kg_fixed_factor','monitoring_seconds',
    'energy_kWh_per_processed_image','time_seconds_per_processed_image',
    'throughput_images_per_second','first_pass_accuracy_pct_diagnostic']
summary_rows=[]
for measure in MEASURES:
    values=results_df[measure].astype(float)
    summary_rows.append({'model':'REMNET','measure':measure,'n_seeds':len(values),
                         'mean':values.mean(),'sample_sd':values.std(ddof=1),
                         'min':values.min(),'max':values.max()})
summary_df=pd.DataFrame(summary_rows)
results_df.to_csv(OUTPUT/'REMNET_Stage3_D3_Inference_Per_Seed.csv',index=False,float_format='%.12f')
summary_df.to_csv(OUTPUT/'REMNET_Stage3_D3_Inference_Five_Seed_Mean_SD.csv',index=False,float_format='%.12f')
print(summary_df[['measure','mean','sample_sd']].to_string(index=False,
    formatters={'mean':lambda v:f'{v:.12f}','sample_sd':lambda v:f'{v:.12f}'}))
print('Five-seed D3 inference summary saved.')


                           measure             mean      sample_sd
        total_inference_energy_kWh   0.000596458457 0.000013397954
       codecarbon_emissions_kgCO2e   0.000151605652 0.000003405443
 standardized_CO2e_kg_fixed_factor   0.000283317767 0.000006364028
                monitoring_seconds  41.057785370400 0.460847842052
    energy_kWh_per_processed_image   0.000000085281 0.000000001916
  time_seconds_per_processed_image   0.005870429707 0.000065891885
      throughput_images_per_second 170.362359215101 1.901969131833
first_pass_accuracy_pct_diagnostic  16.551329711181 2.610889132040
Five-seed D3 inference summary saved.


## 10. Historical Stage 2 constituent training costs (reference only)

The following **previously recorded Stage 2** figures are reproduced from the supplied REMNET training/sustainability report. They count training of **ResNet50 + MobileNetV2 + EfficientNet-Lite0 once per seed**. They are **not newly measured in Stage 3** and are **not added** to the Stage 3 external inference totals. This preserves the required distinction between historical training and current external inference.

The prior Stage 2 run measured inference on 463 images, whereas the Stage 3 D3 workload uses 3,497 external images; those raw session totals are not directly comparable. Historical CO₂e values used the reported fixed factor of **0.475 kg CO₂e/kWh**; Stage 3 saves CodeCarbon's own original emissions separately from standardized values.

**Provenance:** User-supplied `remnet-co2-prob-s2.ipynb` output and five-seed summary. Historical values are reproduced as reported, not re-estimated from the new inference session.

In [14]:
# Historical Stage 2 data supplied with the REMNET sustainability records.
# Do not recompute Stage 2 training from Stage 3 inference or represent it as newly monitored.
historical_training_rows = [
    {'seed':7,   'training_CO2e_kg_reported':0.002162, 'training_energy_kWh_reported':0.004551, 'training_time_min_reported':4.887},
    {'seed':49,  'training_CO2e_kg_reported':0.001810, 'training_energy_kWh_reported':0.003810, 'training_time_min_reported':3.956},
    {'seed':112, 'training_CO2e_kg_reported':0.001727, 'training_energy_kWh_reported':0.003635, 'training_time_min_reported':3.780},
    {'seed':345, 'training_CO2e_kg_reported':0.001737, 'training_energy_kWh_reported':0.003656, 'training_time_min_reported':3.810},
    {'seed':476, 'training_CO2e_kg_reported':0.001824, 'training_energy_kWh_reported':0.003841, 'training_time_min_reported':3.970},
]
historical_training_df = pd.DataFrame(historical_training_rows)
historical_training_df['provenance'] = 'Stage 2 historical report (user supplied; not Stage 3 monitored)'
historical_training_df['constituent_models_counted_once'] = 'ResNet50 + MobileNetV2 + EfficientNet-Lite0'
historical_training_df.to_csv(
    OUTPUT/'REMNET_Stage2_Historical_Constituent_Training_Per_Seed.csv',
    index=False,float_format='%.12f')

# Reproduce exact means as reported by Stage 2, rather than silently replacing
# them by means of individually rounded figures in the displayed table.
reported_training_mean = {
    'training_CO2e_kg_reported':0.001851850,
    'training_energy_kWh_reported':0.003898631,
    'training_time_min_reported':4.080626,
}
historical_summary_rows=[]
for measure in ('training_CO2e_kg_reported','training_energy_kWh_reported','training_time_min_reported'):
    vals=historical_training_df[measure].astype(float)
    historical_summary_rows.append({
        'measure':measure,'mean_as_reported_stage2':reported_training_mean[measure],
        'mean_of_displayed_rounded_seed_rows':float(vals.mean()),
        'sample_sd_from_displayed_seed_rows':float(vals.std(ddof=1)),
        'n_seeds':len(vals),
        'note':'Mean from original Stage 2 summary; SD computed from rounded published seed values'
    })
historical_training_summary_df=pd.DataFrame(historical_summary_rows)
historical_training_summary_df.to_csv(
    OUTPUT/'REMNET_Stage2_Historical_Training_Mean_SD.csv',
    index=False,float_format='%.12f')

print('\n'+'='*75)
print('HISTORICAL STAGE 2 — CONSTITUENT TRAINING (NOT MEASURED IN STAGE 3)')
print('='*75)
print(historical_training_df[['seed','training_CO2e_kg_reported','training_energy_kWh_reported','training_time_min_reported']].to_string(
    index=False,formatters={
        'training_CO2e_kg_reported':lambda v:f'{v:.9f}',
        'training_energy_kWh_reported':lambda v:f'{v:.9f}',
        'training_time_min_reported':lambda v:f'{v:.3f}',
    }))
print('Reported mean training CO2e (kg):',f"{reported_training_mean['training_CO2e_kg_reported']:.9f}")
print('Reported mean training energy (kWh):',f"{reported_training_mean['training_energy_kWh_reported']:.9f}")
print('Reported mean training time (min):',f"{reported_training_mean['training_time_min_reported']:.6f}")
print('Separate from newly measured Stage 3 D3 inference; no combined cost claimed.')

print('\n'+'='*75)
print('STAGE 3 — D3 EXTERNAL INFERENCE (NEWLY MONITORED; 3,497 UNIQUE IMAGES)')
print('='*75)
print(results_df[['seed','unique_external_images','number_complete_repetitions','total_processed_images',
    'total_inference_energy_kWh','codecarbon_emissions_kgCO2e',
    'standardized_CO2e_kg_fixed_factor','monitoring_seconds',
    'energy_kWh_per_processed_image','time_seconds_per_processed_image',
    'throughput_images_per_second']].to_string(index=False,formatters={
       'total_inference_energy_kWh':lambda v:f'{v:.9f}',
       'codecarbon_emissions_kgCO2e':lambda v:f'{v:.9f}',
       'standardized_CO2e_kg_fixed_factor':lambda v:f'{v:.9f}',
       'monitoring_seconds':lambda v:f'{v:.3f}',
       'energy_kWh_per_processed_image':lambda v:f'{v:.12f}',
       'time_seconds_per_processed_image':lambda v:f'{v:.9f}',
       'throughput_images_per_second':lambda v:f'{v:.3f}',
    }))
print('CodeCarbon-emitted CO2e and standardized CO2e at 0.475 kg CO2e/kWh are separate columns.')
print('Five-seed mean standardized CO2e (kg):',
      f"{results_df['standardized_CO2e_kg_fixed_factor'].mean():.9f}")
print('Five-seed sample SD standardized CO2e (kg):',
      f"{results_df['standardized_CO2e_kg_fixed_factor'].std(ddof=1):.9f}")



HISTORICAL STAGE 2 — CONSTITUENT TRAINING (NOT MEASURED IN STAGE 3)
 seed training_CO2e_kg_reported training_energy_kWh_reported training_time_min_reported
    7               0.002162000                  0.004551000                      4.887
   49               0.001810000                  0.003810000                      3.956
  112               0.001727000                  0.003635000                      3.780
  345               0.001737000                  0.003656000                      3.810
  476               0.001824000                  0.003841000                      3.970
Reported mean training CO2e (kg): 0.001851850
Reported mean training energy (kWh): 0.003898631
Reported mean training time (min): 4.080626
Separate from newly measured Stage 3 D3 inference; no combined cost claimed.

STAGE 3 — D3 EXTERNAL INFERENCE (NEWLY MONITORED; 3,497 UNIQUE IMAGES)
 seed  unique_external_images  number_complete_repetitions  total_processed_images total_inference_energy_kWh codec

## 11. Configuration report and one downloadable ZIP archive

In [15]:
configuration={
 'experiment':'REMNET Stage 3 D3 external inference sustainability',
 'source_stage2_sustainability_notebook':'remnet-co2-prob-s2.ipynb',
 'historical_stage2_training':'User-provided Stage 2 constituent training figures; not remeasured or combined with Stage 3 D3 inference',
 'historical_stage2_training_csv':'REMNET_Stage2_Historical_Constituent_Training_Per_Seed.csv',
 'checkpoint_source':'classification checkpoints, REMNET three constituents per seed',
 'input_checkpoint_root':str(REMNET_CKPT_DIR),
 'seeds':list(SEEDS),'class_to_index':CLASS_TO_ID,
 'external_dataset':EXTERNAL_HANDLE,
 'd3_manifest_csv_path':str(EXTERNAL_MANIFEST),
 'external_unique_images':len(external_eval_df),
 'external_manifest_fingerprint':EXTERNAL_FINGERPRINT,
 'd1_original_test_evaluation_performed':False,
 'preprocessing':'RGB, Resize((484,484)), ToTensor(), Normalize((0.5,)*3,(0.5,)*3)',
 'ensemble_architectures':list(CONSTITUENTS),
 'ensemble_rule':'equal-weight average of three constituent softmax probability vectors',
 'no_training_no_adaptation':True,'classification_checkpoints_unmodified':True,
 'checkpoint_loading':'strict state_dict and seed/model metadata checks',
 'batch_size':BATCH_SIZE,'minimum_complete_passes':MIN_COMPLETE_PASSES,
 'minimum_monitor_seconds':MIN_MONITOR_SECONDS,
 'codecarbon_sampling_seconds':CODECARBON_SAMPLING_SECONDS,
 'codecarbon_version':installed_version('codecarbon'),
 'device':str(DEVICE),'hardware_gpu':torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else None,
 'carbon_factor_fixed_kgCO2e_per_kWh':CARBON_FACTOR,
 'measurement_boundary':'Each seed: after model loading and warm-up; repeat complete D3 inference passes, including batch loading/preprocessing; exclude checkpoint loading/warm-up; GPU synchronized for timing.',
 'original_codecarbon_emissions':'Direct CodeCarbon emissions.csv emissions column',
 'standardized_co2':'Energy kWh multiplied by fixed factor, separately reported',
 'generated_utc':datetime.now(timezone.utc).isoformat()
}
(OUTPUT/'REMNET_Stage3_D3_Inference_Configuration.json').write_text(
    json.dumps(configuration,indent=2,ensure_ascii=False),encoding='utf-8')
# ZIP is written outside the results directory so it cannot include itself.
with zipfile.ZipFile(ZIP_PATH,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in sorted(OUTPUT.rglob('*')):
        if p.is_file() and 'original_source_extracted' not in p.parts:
            z.write(p,p.relative_to(OUTPUT))
print('Single downloadable archive:',ZIP_PATH)
print('Archive bytes:',ZIP_PATH.stat().st_size)
print('Outputs: Stage 3 D3 per-seed/mean-SD inference; raw CodeCarbon logs; checkpoint registry; separate historical Stage 2 training tables; configuration JSON.')


Single downloadable archive: /kaggle/working/REMNET_D3_Inference_Sustainability_All_Outputs.zip
Archive bytes: 8957
Outputs: Stage 3 D3 per-seed/mean-SD inference; raw CodeCarbon logs; checkpoint registry; separate historical Stage 2 training tables; configuration JSON.
